# 过滤后的 Liu2023 全量后处理消融

固定同一轮选出的模型、生成种子和逐类测试配额，比较原始生成、仅训练缺失掩码和完整模型。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta
import os, sys, json, gc, hashlib
import numpy as np
import pandas as pd
import anndata as ad
import torch
from threadpoolctl import threadpool_limits

from data.paths import PROJECT, OUTPUTS
OUTPUTS=OUTPUTS/'min10'
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
assert Path(sys.prefix).name.lower()=='simscdnam'
ROOT=OUTPUTS/'ablation/liu2023/full'
ROOT.mkdir(parents=True,exist_ok=True)
(ROOT/'figures').mkdir(exist_ok=True)
RUN=OUTPUTS/'model/liu2023/full'
PREP=OUTPUTS/'preprocessing/liu2023/full'
COLUMNS=['A1 1-WD↓','A1 MSE↓','A1 Pearson↑','A2 Pearson↑','A3 miLISI↑']
torch.set_num_threads(8)
DEVICE=torch.device(os.environ.get('ABLATION_DEVICE', 'cuda:2'))

def ablation_status(phase,**extra):
    record={'阶段':phase,'时间':datetime.now(timezone(timedelta(hours=8))).isoformat(),**extra}
    partial=ROOT/'status.json.partial'
    partial.write_text(json.dumps(record,ensure_ascii=False,indent=2))
    partial.replace(ROOT/'status.json')
    print(record,flush=True)

original_paths=[PREP/'TrainingReal.h5ad',PREP/'HeldoutReal.h5ad',RUN/'CVAE_training_mask/data.h5ad',RUN/'simscDNAm/data.h5ad',RUN/'simscDNAm/model.pt',RUN/'simscDNAm/alignment/metadata.json']
original_stamps={str(path):{'size':path.stat().st_size,'mtime_ns':path.stat().st_mtime_ns} for path in original_paths}
source_files=['model/src/generation.py','model/src/model.py','model/src/postprocessing.py','data/evaluation.py','evaluate/A1/metrics.py','evaluate/A2/metrics.py','evaluate/A3/metrics.py']
source_hashes={name:hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() for name in source_files}
identity={'本轮原始文件':original_stamps,'源码SHA256':source_hashes,'采样种子':42,'近邻数':30}
if (ROOT/'identity.json').exists():
    assert json.loads((ROOT/'identity.json').read_text())==identity
else:
    (ROOT/'identity.json').write_text(json.dumps(identity,ensure_ascii=False,indent=2))
ablation_status('恢复本轮原始生成数据')


In [ ]:
from model.src.model import load_cvae_model
from model.src.generation import write_synthetic_h5ad
from data.training import load_target_labels

configuration=json.loads((RUN/'config.json').read_text())['配置']
raw_path=ROOT/'no_mask_no_alignment/data.h5ad'
training_header=ad.read_h5ad(PREP/'TrainingReal.h5ad',backed='r')
masked_header=ad.read_h5ad(RUN/'CVAE_training_mask/data.h5ad',backed='r')
try:
    model,checkpoint=load_cvae_model(RUN/'simscDNAm/model.pt',DEVICE)
    target=load_target_labels(PREP/'HeldoutReal.h5ad',training_header.var_names.to_numpy(),checkpoint['class_names'])
    np.random.default_rng(configuration['SEED']).shuffle(target)
    np.testing.assert_array_equal(target,masked_header.obs['Subclass'].astype(str).to_numpy())
    lookup={str(name):i for i,name in enumerate(checkpoint['class_names'])}
    classes=np.array([lookup[label] for label in target],dtype=np.int64)
    if not raw_path.exists():
        write_synthetic_h5ad(raw_path,model,classes,target,training_header.var,training_header.var_names.to_numpy(dtype=str),
            checkpoint['sampling_steps'],DEVICE,configuration['SEED'],configuration['GENERATION_ROW_BLOCK'],configuration['FEATURE_BLOCK'])
    del model
    torch.cuda.empty_cache()
finally:
    training_header.file.close()
    masked_header.file.close()
raw=ad.read_h5ad(raw_path,backed='r')
masked=ad.read_h5ad(RUN/'CVAE_training_mask/data.h5ad',backed='r')
try:
    assert raw.shape==masked.shape and raw.var_names.equals(masked.var_names)
    assert raw.obs_names.equals(masked.obs_names)
    for start in range(0,raw.n_obs,16):
        left=np.asarray(raw.X[start:start+16])
        right=np.asarray(masked.X[start:start+16])
        observed=np.isfinite(right)
        assert np.isfinite(left).all() and left.min()>=0 and left.max()<=1
        np.testing.assert_array_equal(left[observed],right[observed])
finally:
    raw.file.close()
    masked.file.close()
ablation_status('原始生成与掩码观测值逐值核验通过')

In [ ]:
from data.evaluation import dense_block,read_labels,validate_region_axis,region_pairs,imputed_features
from evaluate.A1.metrics import a1_metrics
from evaluate.A2.metrics import paired_pearson
from evaluate.A3.metrics import milisi
from sklearn.decomposition import PCA

variant_paths={'同时去掉掩码和对齐':raw_path,'去掉对齐，保留掩码':RUN/'CVAE_training_mask/data.h5ad',
               '完整simscDNAm':RUN/'simscDNAm/data.h5ad'}
with threadpool_limits(limits=8):
    ablation_status('拟合共享训练 PCA')
    real=ad.read_h5ad(PREP/'HeldoutReal.h5ad')
    natural_count=real.n_vars-1000
    with np.load(OUTPUTS/'internal/jobs/liu2023/full/pca.npz') as cache:
        selected,fill=cache['selected'],cache['fill']
        pca=PCA(n_components=len(cache['components']),svd_solver='randomized',random_state=42)
        pca.components_=cache['components']
        pca.mean_=cache['mean']
        pca.n_features_in_=len(selected)
    real_rows=np.arange(real.n_obs)
    real_labels=read_labels(real,real_rows)
    real_rep=pca.transform(imputed_features(real,real_rows,selected,fill))
    gc.collect()
    expected=real.obs['Subclass'].astype(str).value_counts().sort_index()
    main_scores=pd.read_csv(OUTPUTS/'internal/jobs/liu2023/full/simscDNAm/scores.csv',float_precision='round_trip')
    complete_score=main_scores.loc[main_scores['方法'].eq('simscDNAm')].iloc[0]
    full=ad.read_h5ad(variant_paths['完整simscDNAm'])
    full_rep=pca.transform(imputed_features(full,np.arange(full.n_obs),selected,fill))
    full_milisi,full_cells,full_groups=milisi(real_rep,real_labels,full_rep,read_labels(full,np.arange(full.n_obs)),30)
    np.testing.assert_allclose(full_milisi,complete_score[COLUMNS[-1]],rtol=0,atol=1e-12)
    full_cells.insert(0,'方法','完整simscDNAm')
    full_groups.insert(0,'方法','完整simscDNAm')
    full_cells.to_csv(ROOT/'完整模型_A3_cells.csv',index=False)
    full_groups.to_csv(ROOT/'完整模型_A3_types.csv',index=False)
    del full,full_rep
    gc.collect()
    ablation_status('共享训练参照与主实验一致',完整模型miLISI=full_milisi)
    pairs=region_pairs(real,pair_path=PREP/'region_pairs.csv')
    records=[]
    for variant,path in variant_paths.items():
        candidate=ad.read_h5ad(path)
        validate_region_axis(real,candidate)
        assert candidate.n_obs==real.n_obs
        pd.testing.assert_series_equal(candidate.obs['Subclass'].astype(str).value_counts().sort_index(),expected,check_names=False)
        candidate_rows=np.arange(candidate.n_obs)
        if variant=='完整simscDNAm':
            records.append({'方法':variant,**{name:float(complete_score[name]) for name in COLUMNS}})
        else:
            ablation_status('评分',版本=variant)
            a1,details=a1_metrics(real,candidate,real_rows,candidate_rows,natural_count)
            correlations=pd.concat([pairs,paired_pearson(dense_block(candidate,candidate_rows,pairs['原region索引'].to_numpy()),
                dense_block(candidate,candidate_rows,pairs['复制region索引'].to_numpy()))],axis=1)
            rep=pca.transform(imputed_features(candidate,candidate_rows,selected,fill))
            a3,cells,groups=milisi(real_rep,real_labels,rep,read_labels(candidate,candidate_rows),30)
            records.append({'方法':variant,COLUMNS[0]:a1['加权 1-WD'],COLUMNS[1]:a1['加权 MSE'],COLUMNS[2]:a1['加权 Pearson'],
                            COLUMNS[3]:correlations['Pearson'].mean(),COLUMNS[4]:a3})
            index=len(records)
            details.to_csv(ROOT/f'版本{index}_A1_types.csv',index=False)
            correlations.to_csv(ROOT/f'版本{index}_A2_pairs.csv',index=False)
            cells.to_csv(ROOT/f'版本{index}_A3_cells.csv',index=False)
            groups.to_csv(ROOT/f'版本{index}_A3_types.csv',index=False)
            del rep
        pd.DataFrame(records).to_csv(ROOT/'scores.partial.csv',index=False)
        ablation_status('版本评分完成',版本=variant,分数=records[-1])
        del candidate
        gc.collect()
    scores=pd.DataFrame(records)
    assert len(scores)==3 and np.isfinite(scores[COLUMNS].to_numpy()).all()
    scores.to_csv(ROOT/'scores.csv',index=False,encoding='utf-8-sig')
    np.testing.assert_allclose(scores.iloc[-1][COLUMNS].astype(float),complete_score[COLUMNS].astype(float),rtol=0,atol=0)
    del real
    gc.collect()


In [ ]:
for path,stamp in original_stamps.items():
    assert Path(path).stat().st_size == stamp['size'] and Path(path).stat().st_mtime_ns == stamp['mtime_ns'],path
for name,digest in source_hashes.items():
    assert hashlib.sha256((PROJECT/name).read_bytes()).hexdigest() == digest,name
verification = {'状态':'通过','版本数':3,'测试细胞数':len(real_rows),'逐Subclass配额一致':True,
    '原始生成与掩码观测值逐值一致':True,'完整行与主实验分数一致':True,
    '共享本轮训练ANOVA选区':len(selected),'PCA维数':len(pca.components_),'近邻数':30,
    '版本矩阵':{name:str(path) for name,path in variant_paths.items()}}
(ROOT/'verification.json').write_text(json.dumps(verification,ensure_ascii=False,indent=2),encoding='utf-8')
ablation_status('完成',**verification)